# 10 UNMATCHED: sensitivity of classification to onsets without a recovery

Onsets without a matched recovery are kept as zero-duration episodes, so the in-episode exclusion removes only their onset second. This notebook asks whether that treatment affects the reported scores.

The primary XGBoost specification (17 measurement channels, default class weighting, two-stage fit with early stopping on validation, seed 42) is fitted to three label sets:

* **primary**: the labels used in the paper;
* **A**: unmatched onsets removed;
* **B**: unmatched episodes given the median matched duration of their message.

Each label set is scored for the pooled target and for the two fault groups of Table 10 (the blower message alone; the other messages), built as in `11_CHECK_BLOWER`. The model code is the same as in `07_CLASSIFY` and `11_CHECK_BLOWER`; the label variants come from `03_LABELS`. Output: `OUTPUT/v4/rev_unmatched_results.csv`.

Paper: Sections 2.7.1 and 3.2.

In [ ]:
import os, sys, warnings, time
import numpy as np, pandas as pd
import xgboost
from sklearn.metrics import average_precision_score, precision_recall_curve
from xgboost import XGBClassifier
warnings.filterwarnings("ignore")

A2 = "../ANALYSIS2"
CFG = dict(
    labelled   = os.environ.get("REV_BUILDDIR", "build") + "/telemetry_labelled_v4.pkl",
    episodes   = os.environ.get("REV_BUILDDIR", "build") + "/fault_episodes_v4.csv",
    ref_pooled = f"{A2}/OUTPUT/classification_results_v3.csv",
    ref_groups = f"{A2}/OUTPUT/blower_performance_check.csv",
    variants   = os.environ.get("REV_BUILDDIR", "build") + "/labels_variants_june_v4.pkl",
    outdir     = os.environ.get("REV_OUTDIR", "OUTPUT") + "/v4",
    blower     = "AUX CAP MACH RM BLOW1 NOT OFF",
    horizons   = os.environ.get("REV_HORIZONS", "D5,W5,W60,W300").split(","),
    windows    = {"D5": (-5, 5), "W5": (-5, -1), "W60": (-60, -1), "W300": (-300, -1)},
    patience   = 100, test_frac = 0.20, val_frac = 0.20, seed = 42,
    recall_points = [0.25, 0.50],
)
NUMERIC = ["KMH", "DIST_M", "KV", "AMP", "KWH", "BV", "PRESSURE", "TRACTION_PCT", "BG1_KN", "BG2_KN", "BG1_DCLV", "BG2_DCLV"]
WHEEL_SLIP = ["Wheel_Slip_TM2", "Wheel_Slip_TM3", "Wheel_Slip_TM4", "Wheel_Slip_TM5", "Wheel_Slip_TM6"]
PRIMARY = NUMERIC + WHEEL_SLIP
os.makedirs(CFG["outdir"], exist_ok=True)
print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| xgboost", xgboost.__version__)

## 1. Data, labels and onsets

In [ ]:
tel = pd.read_pickle(CFG["labelled"]).sort_values("DATETIME").reset_index(drop=True)
var = pd.read_pickle(CFG["variants"])
assert np.array_equal(var["DATETIME"].values, tel["DATETIME"].values), "variant labels not aligned"
for h in CFG["horizons"]:
    tel[f"{h}_A"], tel[f"{h}_B"] = var[f"{h}_A"].values, var[f"{h}_B"].values
del var
if os.environ.get("REV_SAMPLE"):          # trial runs only: first part of the month
    tel = tel.iloc[:int(len(tel) * float(os.environ["REV_SAMPLE"]))].copy()
    print(f"TRIAL RUN on {len(tel):,} rows")

t0, t1 = tel["DATETIME"].min(), tel["DATETIME"].max()
ep = pd.read_csv(CFG["episodes"], parse_dates=["onset", "recovery"])
ep = ep[(ep.onset >= t0) & (ep.onset <= t1)].copy()
ep["is_blower"] = ep["fault"].eq(CFG["blower"])
ONSETS = {"primary": ep, "A": ep[ep.matched], "B": ep}
for k, e in ONSETS.items():
    print(f"{k:8s} onsets {len(e):4d}  blower {int(e.is_blower.sum()):3d}  other {int((~e.is_blower).sum()):3d}")

X_all = tel[PRIMARY].astype("float32").fillna(0).values
ts = tel["DATETIME"].values.astype("datetime64[s]")

## 2. Model code

`three_way_split` is the partition of `07_CLASSIFY`; `xgb` and `fit_score` are the two-stage XGBoost fit of `11_CHECK_BLOWER` (equivalent to the default-weighting XGBoost of `07_CLASSIFY`); `coverage` builds the fault-group targets.

In [ ]:
def chrono_cut(y, frac):
    n = len(y); i = int(n * (1 - frac))
    while i < n - 1 and y[i] == 1 and y[i - 1] == 1:
        i += 1
    return i

def three_way_split(y):
    n = len(y); t = int(n * (1 - CFG["test_frac"]))
    while t < n - 1 and y[t] == 1:
        t += 1
    v = chrono_cut(y[:t], CFG["val_frac"])
    return np.arange(v), np.arange(v, t), np.arange(t), np.arange(t, n)

def xgb(y, n_iter=None):
    spw = float((y == 0).sum() / max((y == 1).sum(), 1))
    kw = dict(n_estimators=600, early_stopping_rounds=CFG["patience"]) if n_iter is None else dict(n_estimators=n_iter)
    return XGBClassifier(learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8,
                         eval_metric="aucpr", tree_method="hist", scale_pos_weight=spw,
                         random_state=CFG["seed"], n_jobs=-1, **kw)

def fit_score(X, y):
    ia, iv, itr, ite = three_way_split(y)
    m1 = xgb(y[ia]).fit(X[ia], y[ia], eval_set=[(X[iv], y[iv])], verbose=False)
    n_iter = int(m1.best_iteration) + 1
    m2 = xgb(y[itr], n_iter).fit(X[itr], y[itr])
    s = m2.predict_proba(X[ite])[:, 1]
    base = y[ite].mean()
    ap = average_precision_score(y[ite], s)
    p, r, _ = precision_recall_curve(y[ite], s)
    out = dict(pr_auc=ap, baseline=base, lift=ap / base,
               test_positive_s=int(y[ite].sum()), trees=n_iter)
    for rp in CFG["recall_points"]:
        out[f"precision_at_recall_{rp:.2f}"] = float(p[r >= rp].max())
    return out

def coverage(onsets, lo, hi):
    d = np.zeros(len(ts) + 1, dtype=np.int32)
    o = pd.DatetimeIndex(onsets).values.astype("datetime64[s]")
    a = np.searchsorted(ts, o + np.timedelta64(lo, "s"), side="left")
    b = np.searchsorted(ts, o + np.timedelta64(hi, "s"), side="right")
    np.add.at(d, a, 1); np.add.at(d, b, -1)
    return np.cumsum(d)[:-1]

def targets(h, variant):
    # pooled, blower-only and other-label targets for one window and one label set
    y0 = tel[h if variant == "primary" else f"{h}_{variant}"].astype(int).values
    e = ONSETS[variant]
    lo, hi = CFG["windows"][h]
    cov_b = coverage(e.onset[e.is_blower], lo, hi)
    cov_o = coverage(e.onset[~e.is_blower], lo, hi)
    return {"pooled":      y0,
            "other labels": np.where((y0 == 1) & (cov_b > 0) & (cov_o == 0), -1, y0),
            "blower only":  np.where((y0 == 1) & (cov_o > 0) & (cov_b == 0), -1, y0)}

## 3. Primary labels

The primary labels are fitted first. A comparison with the results of the originally submitted manuscript is printed for reference only, since those rest on an earlier version of the fault log.

In [ ]:
ref_p = (pd.read_csv(CFG["ref_pooled"]).query("model == 'XGBoost' and sampling == 'Default (class weight)'").set_index("horizon"))
ref_g = pd.read_csv(CFG["ref_groups"]).set_index(["horizon", "variant"])
gmap = {"other labels": "Without blower", "blower only": "Blower only"}

rows = []
t_start = time.time()
for h in CFG["horizons"]:
    for target, y in targets(h, "primary").items():
        keep = y != -1
        r = fit_score(X_all[keep], y[keep])
        r.update(horizon=h, labels="primary", target=target)
        if target == "pooled":
            r["reference"] = ref_p.loc[h, "pr_auc"]
            r["reproduced"] = abs(r["pr_auc"] - r["reference"]) < 1e-9
        else:
            r["reference"] = ref_g.loc[(h, gmap[target]), "pr_auc"]
            r["reproduced"] = round(r["pr_auc"], 3) == r["reference"]
        rows.append(r)
        print(f"{h:5s} {target:13s} PR-AUC {r['pr_auc']:.4f}  reference {r['reference']:.4f}  "
              f"{'same' if r['reproduced'] else 'differs'}  trees {r['trees']}")
prim = pd.DataFrame(rows)
print(f"[{time.time() - t_start:.0f} s]")
if True:   # corrected log: comparison with the submitted results only
    # trial run on another platform: XGBoost row subsampling draws differ between
    # Windows and Linux builds, so exact reproduction is expected only on the original Windows machine
    print("corrected log: results of the submitted manuscript shown for comparison only")
else:
    assert prim.reproduced.all(), "submitted figures not reproduced; stop and investigate"

## 4. Variants A and B

In [ ]:
rows = []
t_start = time.time()
for v in ["A", "B"]:
    for h in CFG["horizons"]:
        for target, y in targets(h, v).items():
            keep = y != -1
            r = fit_score(X_all[keep], y[keep])
            r.update(horizon=h, labels=v, target=target)
            rows.append(r)
            print(f"{v} {h:5s} {target:13s} PR-AUC {r['pr_auc']:.4f}  lift {r['lift']:.1f}x  trees {r['trees']}")
print(f"[{time.time() - t_start:.0f} s]")
res = pd.concat([prim.drop(columns=["reference", "reproduced"]), pd.DataFrame(rows)], ignore_index=True)
res.to_csv(f"{CFG['outdir']}/rev_unmatched_results.csv", index=False)

## 5. Comparison

In [ ]:
order = ["pooled", "blower only", "other labels"]
for metric, fmt in [("pr_auc", "{:.3f}"), ("lift", "{:.1f}")]:
    t = res.pivot_table(index=["target", "labels"], columns="horizon", values=metric)
    t = t.reindex(columns=CFG["horizons"]).reindex(order, level=0)
    print(f"--- {metric}"); print(t.map(fmt.format).to_string(), "\n")

d = res.pivot_table(index=["target", "horizon"], columns="labels", values="pr_auc")
d["A minus primary"] = d["A"] - d["primary"]
d["B minus primary"] = d["B"] - d["primary"]
d = d.reindex(order, level=0)
print("--- change in PR-AUC"); print(d.round(3).to_string())
print(f"\nLargest absolute change: {d[['A minus primary', 'B minus primary']].abs().max().max():.3f}")

___
**END**